In [92]:
import pandas as pd
import numpy as np

In [93]:
path = "../data/df_diabetic.csv"

df = pd.read_csv(path, low_memory=False)

df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,No,No,No,No,No,No,No,No,Yes,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,No,Steady,No,No,No,No,No,Ch,Yes,0


## Feature Engineering

In [94]:
df["severity"] = df["time_in_hospital"] * (df["number_diagnoses"] + 1)

In [95]:
df['total_visits'] = (
    df['number_inpatient'] +
    df['number_outpatient'] +
    df['number_emergency']
)

Grouping ICD9 codes (diag_[1,2,3])

In [96]:
def match_icd9_group(diag_col):
    #ICD9 codes referring to an external cause begin with a letter E-V. I'll replace all of these with numeric values using regex
    diag_col = diag_col.str.replace(r'[E-V]\d*', '1000', regex = True).astype(float)

    #Matches each condition with its ICD9 grouping
    conditions = [
        (diag_col.between(390, 459)) | (diag_col == 785), #circulatory
        (diag_col.between(460, 519)) | (diag_col == 786), #respiratory
        (diag_col.between(520, 579)) | (diag_col == 787), #digestive
        np.floor(diag_col) == 250, #diabetes
        diag_col.between(800, 999), #injury
        diag_col.between(710, 739), #musculoskeletal
        (diag_col.between(580, 629)) | (diag_col == 788), #genitourinary
        diag_col.between(140, 239), #neoplasms (cancer/other tissue growths)
        diag_col.isna() #na
    ] #anything else is defined as 'other'

    groups = [
        'circulatory', 
        'respiratory', 
        'digestive', 
        'diabetes', 
        'injury', 
        'musculoskeletal',
        'genitourinary',
        'neoplasms',
        'na'
    ]
    
    return pd.Series(np.select(conditions, groups, default = 'other'))

In [97]:
df['diag_1g'] = match_icd9_group(df['diag_1'])
df['diag_2g'] = match_icd9_group(df['diag_2'])
df['diag_3g'] = match_icd9_group(df['diag_3'])

df.drop(['diag_1', 'diag_2', 'diag_3'], axis=1, inplace=True)
df = df[df['diag_1g'] != 'na']

Grouping Rare Medical Specialties

In [98]:
specialty_counts = df['medical_specialty'].value_counts()

rare_specialties = specialty_counts[
    specialty_counts < 500
].index

df['medical_specialty'] = df['medical_specialty'].replace(
    rare_specialties,
    'Other'
)

In [104]:
df['medical_specialty'].dropna().unique()

array(['Other', 'InternalMedicine', 'Family/GeneralPractice',
       'Cardiology', 'Surgery-General', 'Orthopedics', 'Gastroenterology',
       'Surgery-Cardiovascular/Thoracic', 'Nephrology',
       'Orthopedics-Reconstructive', 'Psychiatry', 'Emergency/Trauma',
       'Pulmonology', 'ObstetricsandGynecology', 'Urology',
       'Surgery-Vascular', 'Radiologist'], dtype=object)

In [99]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted',
       'severity', 'total_visits', 'diag_1g', 'diag_2g', 'diag_3g'],
      dtype='object')

## Feature Selection

In [100]:
corr = df.corr(numeric_only=True)["readmitted"].sort_values(ascending=False)
print(corr)

readmitted                  1.000000
number_inpatient            0.165115
total_visits                0.126065
number_emergency            0.060697
discharge_disposition_id    0.050613
severity                    0.050259
number_diagnoses            0.049506
time_in_hospital            0.044208
num_medications             0.038447
num_lab_procedures          0.020446
number_outpatient           0.018881
patient_nbr                 0.007847
admission_source_id         0.005887
encounter_id               -0.008631
admission_type_id          -0.011589
num_procedures             -0.012138
Name: readmitted, dtype: float64


# Saving

In [101]:
df.to_csv("../data/df_model.csv", index=False)